In [3]:
import pandas as pd
import numpy as np
import string
import emoji
import re
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
import spacy
from better_profanity import profanity
profanity.load_censor_words()

# Setup
nltk.download("stopwords")
stopwords = set(nltk.corpus.stopwords.words("english"))
nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()

# Load CSV
df = pd.read_csv("cleaned_cyberbullying.csv").dropna(subset=["tweet_text"])
texts = df["tweet_text"].astype(str)

def count_profanity(text):
    return sum(1 for word in text.split() if profanity.contains_profanity(word))

# Feature extraction function
def extract_features(text):
    blob = TextBlob(text)
    vader_scores = vader.polarity_scores(text)
    words = text.split()
    char_count = len(text)
    word_count = len(words)
    punctuation_count = sum(1 for c in text if c in string.punctuation)
    capital_words = [w for w in words if w.isupper() and len(w) > 1]
    exclamations = text.count("!")
    questions = text.count("?")
    mentions = text.count("@")
    hashtags = text.count("#")
    emojis = emoji.emoji_count(text)
    badword_hits = count_profanity(text)
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    uppercase_ratio = sum(1 for c in text if c.isupper()) / char_count if char_count else 0
    stopword_ratio = sum(1 for w in words if w.lower() in stopwords) / word_count if word_count else 0
    repeated_chars = len(re.findall(r"(.)\1{2,}", text))
    
    # POS counts
    doc = nlp(text)
    pos_counts = doc.count_by(spacy.attrs.POS)
    noun_count = pos_counts.get(nlp.vocab.strings["NOUN"], 0)
    verb_count = pos_counts.get(nlp.vocab.strings["VERB"], 0)
    adj_count = pos_counts.get(nlp.vocab.strings["ADJ"], 0)
    adv_count = pos_counts.get(nlp.vocab.strings["ADV"], 0)

    return {
        "char_count": char_count,
        "word_count": word_count,
        "unique_word_count": len(set(words)),
        "punctuation_count": punctuation_count,
        "capital_word_count": len(capital_words),
        "uppercase_ratio": uppercase_ratio,
        "exclamation_count": exclamations,
        "question_count": questions,
        "mention_count": mentions,
        "hashtag_count": hashtags,
        "emoji_count": emojis,
        "badword_count": badword_hits,
        "avg_word_length": avg_word_len,
        "stopword_ratio": stopword_ratio,
        "repeated_char_sequences": repeated_chars,
        "sentiment_polarity": blob.sentiment.polarity,
        "sentiment_subjectivity": blob.sentiment.subjectivity,
        "vader_neg": vader_scores["neg"],
        "vader_neu": vader_scores["neu"],
        "vader_pos": vader_scores["pos"],
        "vader_compound": vader_scores["compound"],
        "noun_count": noun_count,
        "verb_count": verb_count,
        "adj_count": adj_count,
        "adv_count": adv_count,
    }

# Apply to all rows
feature_df = texts.apply(lambda x: pd.Series(extract_features(x)))
df_extended = pd.concat([df.reset_index(drop=True), feature_df], axis=1)

# Save if needed
df_extended.to_csv("cyberbullying_tweets_enriched.csv", index=False)

# Preview
print(df_extended.head())


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jdxli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                                          tweet_text cyberbullying_type  \
0  In other words #katandandre, your food was cra...  not_cyberbullying   
1  Why is #aussietv so white? #MKR #theblock #ImA...  not_cyberbullying   
2  @XochitlSuckkks a classy whore? Or more red ve...  not_cyberbullying   
3  @Jason_Gio meh. :P  thanks for the heads up, b...  not_cyberbullying   
4  @RudhoeEnglish This is an ISIS account pretend...  not_cyberbullying   

                                       cleaned_words  char_count  word_count  \
0             word katandandre food crapilicious mkr        61.0         9.0   
1  aussietv white mkr theblock imacelebrityau tod...       115.0        14.0   
2     xochitlsuckkks classy whore red velvet cupcake        60.0         9.0   
3  jason_gio meh p thanks head concerned another ...       103.0        18.0   
4  rudhoeenglish isi account pretending kurdish a...       103.0        18.0   

   unique_word_count  punctuation_count  capital_word_count  upperca

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import joblib

# Load the enriched dataset
df = pd.read_csv("cyberbullying_tweets_enriched.csv")

# Encode the labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["cyberbullying_type"])
joblib.dump(label_encoder, "rf_label_encoder.pkl")

# Define features to use (exclude text + original label columns)
excluded_columns = ["tweet_text", "cyberbullying_type", "label", "cleaned_words"]
feature_columns = [col for col in df.columns if col not in excluded_columns]

X = df[feature_columns]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)
rf.fit(X_train, y_train)
joblib.dump(rf, "rf_model.pkl")
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))


                     precision    recall  f1-score   support

                age       0.67      0.69      0.68      1598
          ethnicity       0.83      0.66      0.74      1592
             gender       0.66      0.60      0.63      1590
  not_cyberbullying       0.45      0.50      0.47      1587
other_cyberbullying       0.46      0.52      0.49      1565
           religion       0.65      0.66      0.65      1600

           accuracy                           0.61      9532
          macro avg       0.62      0.61      0.61      9532
       weighted avg       0.62      0.61      0.61      9532



In [11]:
import pandas as pd
import numpy as np
import string
import emoji
import re
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import nltk
import spacy
from better_profanity import profanity
profanity.load_censor_words()
import joblib

# Setup
nltk.download("stopwords")
stopwords = set(nltk.corpus.stopwords.words("english"))
nlp = spacy.load("en_core_web_sm")
vader = SentimentIntensityAnalyzer()

# Assuming you reuse your same feature extraction function
new_text = "you dumb nigger."

from transformers import AutoTokenizer
# use your previously defined extract_features() function from earlier

def count_profanity(text):
    return sum(1 for word in text.split() if profanity.contains_profanity(word))

def extract_features(text):
    blob = TextBlob(text)
    vader_scores = vader.polarity_scores(text)
    words = text.split()
    char_count = len(text)
    word_count = len(words)
    punctuation_count = sum(1 for c in text if c in string.punctuation)
    capital_words = [w for w in words if w.isupper() and len(w) > 1]
    exclamations = text.count("!")
    questions = text.count("?")
    mentions = text.count("@")
    hashtags = text.count("#")
    emojis = emoji.emoji_count(text)
    badword_hits = count_profanity(text)
    avg_word_len = np.mean([len(w) for w in words]) if words else 0
    uppercase_ratio = sum(1 for c in text if c.isupper()) / char_count if char_count else 0
    stopword_ratio = sum(1 for w in words if w.lower() in stopwords) / word_count if word_count else 0
    repeated_chars = len(re.findall(r"(.)\1{2,}", text))
    
    # POS counts
    doc = nlp(text)
    pos_counts = doc.count_by(spacy.attrs.POS)
    noun_count = pos_counts.get(nlp.vocab.strings["NOUN"], 0)
    verb_count = pos_counts.get(nlp.vocab.strings["VERB"], 0)
    adj_count = pos_counts.get(nlp.vocab.strings["ADJ"], 0)
    adv_count = pos_counts.get(nlp.vocab.strings["ADV"], 0)

    return {
        "char_count": char_count,
        "word_count": word_count,
        "unique_word_count": len(set(words)),
        "punctuation_count": punctuation_count,
        "capital_word_count": len(capital_words),
        "uppercase_ratio": uppercase_ratio,
        "exclamation_count": exclamations,
        "question_count": questions,
        "mention_count": mentions,
        "hashtag_count": hashtags,
        "emoji_count": emojis,
        "badword_count": badword_hits,
        "avg_word_length": avg_word_len,
        "stopword_ratio": stopword_ratio,
        "repeated_char_sequences": repeated_chars,
        "sentiment_polarity": blob.sentiment.polarity,
        "sentiment_subjectivity": blob.sentiment.subjectivity,
        "vader_neg": vader_scores["neg"],
        "vader_neu": vader_scores["neu"],
        "vader_pos": vader_scores["pos"],
        "vader_compound": vader_scores["compound"],
        "noun_count": noun_count,
        "verb_count": verb_count,
        "adj_count": adj_count,
        "adv_count": adv_count,
    }

new_row = pd.Series(extract_features(new_text)).to_frame().T
new_row.fillna(0, inplace=True)  # in case any features are missing

# Load model + encoder
rf = joblib.load("rf_model.pkl")
label_encoder = joblib.load("rf_label_encoder.pkl")

# Predict
pred_label = label_encoder.inverse_transform(rf.predict(new_row))[0]
print(f"Predicted class: {pred_label}")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jdxli\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Predicted class: not_cyberbullying
